In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import cv2
import tkinter as tk
from tkinter import filedialog, simpledialog
from datetime import datetime
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_rate=0.2):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        dropout_rate=0.2
    ):
        super().__init__()

        self.conv1 = DoubleConv(
            in_channels,
            64,
            dropout_rate
        )
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = DoubleConv(
            64,
            128,
            dropout_rate
        )
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = DoubleConv(
            128,
            256,
            dropout_rate
        )
        self.pool3 = nn.MaxPool2d(2)

        self.conv4 = DoubleConv(
            256,
            512,
            dropout_rate
        )
        self.pool4 = nn.MaxPool2d(2)

        self.conv5 = DoubleConv(
            512,
            1024,
            dropout_rate
        )

        self.up6 = nn.ConvTranspose2d(
            1024,
            512,
            kernel_size=2,
            stride=2
        )

        self.conv6 = DoubleConv(
            1024,
            512,
            dropout_rate
        )

        self.up7 = nn.ConvTranspose2d(
            512,
            256,
            kernel_size=2,
            stride=2
        )

        self.conv7 = DoubleConv(
            512,
            256,
            dropout_rate
        )

        self.up8 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.conv8 = DoubleConv(
            256,
            128,
            dropout_rate
        )

        self.up9 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.conv9 = DoubleConv(
            128,
            64,
            dropout_rate
        )

        self.conv10 = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        conv1 = self.conv1(x)
        pool1 = self.pool1(conv1)

        conv2 = self.conv2(pool1)
        pool2 = self.pool2(conv2)

        conv3 = self.conv3(pool2)
        pool3 = self.pool3(conv3)

        conv4 = self.conv4(pool3)
        pool4 = self.pool4(conv4)

        conv5 = self.conv5(pool4)

        up6 = self.up6(conv5)
        merge6 = torch.cat(
            [up6, conv4],
            dim=1
        )
        conv6 = self.conv6(merge6)

        up7 = self.up7(conv6)
        merge7 = torch.cat(
            [up7, conv3],
            dim=1
        )
        conv7 = self.conv7(merge7)

        up8 = self.up8(conv7)
        merge8 = torch.cat(
            [up8, conv2],
            dim=1
        )
        conv8 = self.conv8(merge8)

        up9 = self.up9(conv8)
        merge9 = torch.cat(
            [up9, conv1],
            dim=1
        )
        conv9 = self.conv9(merge9)

        return self.conv10(conv9)


def select_model():
    root = tk.Tk()
    root.withdraw()

    model_path = filedialog.askopenfilename(
        title="Select Final Trained Model",
        filetypes=[
            ("PyTorch Model", "*.pth")
        ]
    )

    root.destroy()

    return model_path


def select_image():
    root = tk.Tk()
    root.withdraw()

    image_path = filedialog.askopenfilename(
        title="Select TEM Image",
        filetypes=[
            (
                "TEM Images",
                "*.tif *.tiff *.png *.jpg *.jpeg"
            )
        ]
    )

    root.destroy()

    return image_path


def select_output_directory():
    root = tk.Tk()
    root.withdraw()

    output_dir = filedialog.askdirectory(
        title="Select Output Directory"
    )

    root.destroy()

    return output_dir


def get_thresholds():

    root = tk.Tk()
    root.withdraw()

    threshold_input = simpledialog.askstring(
        "Probability Threshold",
        "Enter a probability threshold between 0 and 1.\n\n"
        "Leave blank to generate masks at:\n"
        "0.00, 0.25, 0.50, 0.75 and 1.00"
    )

    root.destroy()

    if (
        threshold_input is None
        or threshold_input.strip() == ""
    ):
        return np.array([
            0.00,
            0.25,
            0.50,
            0.75,
            1.00
        ])

    try:

        threshold = float(
            threshold_input.strip()
        )

        if not 0 <= threshold <= 1:
            raise ValueError

        return np.array([
            threshold
        ])

    except ValueError:

        print(
            "Invalid threshold. "
            "Please enter a value between 0 and 1."
        )

        return get_thresholds()


def load_model(model_path):

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = UNet(
        in_channels=1,
        out_channels=1,
        dropout_rate=0.2
    ).to(device)

    try:

        checkpoint = torch.load(
            model_path,
            map_location=device,
            weights_only=False
        )

    except TypeError:

        checkpoint = torch.load(
            model_path,
            map_location=device
        )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        print(
            "Final trained model loaded."
        )

        if "epoch" in checkpoint:
            print(
                f"Training epoch: "
                f"{checkpoint['epoch']}"
            )

        if "val_loss" in checkpoint:
            print(
                f"Validation loss: "
                f"{checkpoint['val_loss']}"
            )

    else:

        model.load_state_dict(
            checkpoint
        )

        print(
            "Model weights loaded."
        )

    model.eval()

    return model, device


def predict_features(
    model,
    image_path,
    device
):

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Could not read image:\n"
            f"{image_path}"
        )

    original_image = image.copy()

    image_resized = cv2.resize(
        image,
        (1024, 1024),
        interpolation=cv2.INTER_AREA
    )

    image_normalized = (
        image_resized.astype(
            np.float32
        ) / 255.0
    )

    image_tensor = (
        torch.from_numpy(
            image_normalized
        )
        .unsqueeze(0)
        .unsqueeze(0)
        .to(device)
    )

    with torch.no_grad():

        output = model(
            image_tensor
        )

        probability = torch.sigmoid(
            output
        )

        probability = (
            probability
            .cpu()
            .numpy()[0, 0]
        )

    height, width = (
        original_image.shape
    )

    probability_resized = cv2.resize(
        probability,
        (width, height),
        interpolation=cv2.INTER_LINEAR
    )

    return (
        original_image,
        probability_resized
    )


def create_binary_mask(
    probability,
    threshold
):

    return (
        probability >= threshold
    ).astype(
        np.uint8
    )


def analyze_features(
    binary_mask,
    min_area=5
):

    mask_uint8 = (
        binary_mask * 255
    ).astype(
        np.uint8
    )

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    num_features = len(
        contours
    )

    areas = [
        cv2.contourArea(
            contour
        )
        for contour in contours
    ]

    valid_areas = [
        area
        for area in areas
        if area > min_area
    ]

    total_pixels = (
        binary_mask.shape[0]
        * binary_mask.shape[1]
    )

    if valid_areas:

        average_area = np.mean(
            valid_areas
        )

        total_area = np.sum(
            valid_areas
        )

        area_fraction = (
            total_area
            / total_pixels
            * 100.0
        )

    else:

        average_area = 0.0
        total_area = 0.0
        area_fraction = 0.0

    return {
        "num_features": num_features,
        "valid_features": len(
            valid_areas
        ),
        "average_area": average_area,
        "total_area": total_area,
        "area_fraction": area_fraction
    }


def create_overlay(
    image,
    binary_mask,
    alpha=0.6
):

    rgb_image = cv2.cvtColor(
        image,
        cv2.COLOR_GRAY2RGB
    )

    overlay_colour = np.zeros_like(
        rgb_image
    )

    overlay_colour[
        binary_mask > 0
    ] = [
        255,
        0,
        0
    ]

    overlay = cv2.addWeighted(
        rgb_image,
        1.0,
        overlay_colour,
        alpha,
        0
    )

    return overlay


def save_probability_map(
    probability,
    output_path
):

    probability_image = (
        probability * 255.0
    ).clip(
        0,
        255
    ).astype(
        np.uint8
    )

    cv2.imwrite(
        output_path,
        probability_image
    )


def save_binary_mask(
    binary_mask,
    output_path
):

    cv2.imwrite(
        output_path,
        (
            binary_mask * 255
        ).astype(
            np.uint8
        )
    )


def create_visualization(
    original_image,
    probability,
    threshold,
    binary_mask,
    metrics,
    output_path
):

    original_normalized = (
        original_image.astype(
            np.float32
        ) / 255.0
    )

    overlay = create_overlay(
        original_image,
        binary_mask
    )

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 10)
    )

    axes[0, 0].imshow(
        original_normalized,
        cmap="gray"
    )

    axes[0, 0].set_title(
        "Original TEM Image"
    )

    axes[0, 0].axis("off")

    probability_plot = axes[0, 1].imshow(
        probability,
        cmap="hot",
        vmin=0,
        vmax=1
    )

    axes[0, 1].set_title(
        "Feature Probability Map"
    )

    axes[0, 1].axis("off")

    fig.colorbar(
        probability_plot,
        ax=axes[0, 1],
        fraction=0.046,
        pad=0.04,
        label="Probability"
    )

    axes[1, 0].imshow(
        binary_mask,
        cmap="gray"
    )

    axes[1, 0].set_title(
        f"Binary Mask "
        f"(Threshold = {threshold:.2f})"
    )

    axes[1, 0].axis("off")

    axes[1, 1].imshow(
        overlay
    )

    axes[1, 1].set_title(
        "Feature Overlay"
    )

    axes[1, 1].axis("off")

    fig.text(
        0.5,
        0.015,
        f"Features: "
        f"{metrics['num_features']}    |    "
        f"Valid features: "
        f"{metrics['valid_features']}    |    "
        f"Average area: "
        f"{metrics['average_area']:.2f} pixels    |    "
        f"Area fraction: "
        f"{metrics['area_fraction']:.2f}%",
        ha="center",
        fontsize=10
    )

    plt.tight_layout(
        rect=[
            0,
            0.05,
            1,
            1
        ]
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


def save_metrics(
    metrics,
    threshold,
    image_name,
    model_path,
    output_path
):

    with open(
        output_path,
        "w"
    ) as file:

        file.write(
            "Feature Analysis Results\n"
        )

        file.write(
            f"Image: {image_name}\n"
        )

        file.write(
            f"Model: "
            f"{os.path.basename(model_path)}\n"
        )

        file.write(
            f"Threshold: {threshold:.4f}\n"
        )

        file.write(
            f"Detected features: "
            f"{metrics['num_features']}\n"
        )

        file.write(
            f"Valid features: "
            f"{metrics['valid_features']}\n"
        )

        file.write(
            f"Average area: "
            f"{metrics['average_area']:.4f} pixels\n"
        )

        file.write(
            f"Total area: "
            f"{metrics['total_area']:.4f} pixels\n"
        )

        file.write(
            f"Area fraction: "
            f"{metrics['area_fraction']:.4f}%\n"
        )


def save_threshold_summary(
    all_results,
    output_path
):

    with open(
        output_path,
        "w"
    ) as file:

        file.write(
            "threshold,num_features,"
            "valid_features,average_area,"
            "total_area,area_fraction\n"
        )

        for threshold, metrics in all_results:

            file.write(
                f"{threshold:.4f},"
                f"{metrics['num_features']},"
                f"{metrics['valid_features']},"
                f"{metrics['average_area']:.4f},"
                f"{metrics['total_area']:.4f},"
                f"{metrics['area_fraction']:.4f}\n"
            )


def main():

    print(
        "CrispTEM Feature Detector"
    )

    print(
        "=========================="
    )

    model_path = select_model()

    if not model_path:

        print(
            "No model selected. Exiting."
        )

        return

    print(
        f"Selected model:\n{model_path}"
    )

    image_path = select_image()

    if not image_path:

        print(
            "No TEM image selected. Exiting."
        )

        return

    print(
        f"Selected TEM image:\n{image_path}"
    )

    output_base = select_output_directory()

    if not output_base:

        print(
            "No output directory selected. Exiting."
        )

        return

    thresholds = get_thresholds()

    image_name = os.path.splitext(
        os.path.basename(
            image_path
        )
    )[0]

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    output_dir = os.path.join(
        output_base,
        f"feature_analysis_"
        f"{image_name}_"
        f"{timestamp}"
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    print(
        "\nLoading trained model..."
    )

    model, device = load_model(
        model_path
    )

    print(
        f"Using device: {device}"
    )

    if torch.cuda.is_available():

        print(
            f"GPU: "
            f"{torch.cuda.get_device_name(0)}"
        )

    else:

        print(
            "GPU not available. Using CPU."
        )

    print(
        "\nGenerating probability map..."
    )

    (
        original_image,
        probability
    ) = predict_features(
        model,
        image_path,
        device
    )

    probability_path = os.path.join(
        output_dir,
        f"{image_name}_probability_map.png"
    )

    save_probability_map(
        probability,
        probability_path
    )

    print(
        f"Probability map saved to:\n"
        f"{probability_path}"
    )

    all_results = []

    for threshold in thresholds:

        print(
            f"\nProcessing threshold: "
            f"{threshold:.2f}"
        )

        binary_mask = create_binary_mask(
            probability,
            threshold
        )

        metrics = analyze_features(
            binary_mask
        )

        all_results.append(
            (
                threshold,
                metrics
            )
        )

        threshold_string = (
            f"{threshold:.2f}"
            .replace(
                ".",
                "_"
            )
        )

        mask_path = os.path.join(
            output_dir,
            f"{image_name}_mask_"
            f"threshold_{threshold_string}.png"
        )

        save_binary_mask(
            binary_mask,
            mask_path
        )

        visualization_path = os.path.join(
            output_dir,
            f"{image_name}_analysis_"
            f"threshold_{threshold_string}.png"
        )

        create_visualization(
            original_image,
            probability,
            threshold,
            binary_mask,
            metrics,
            visualization_path
        )

        metrics_path = os.path.join(
            output_dir,
            f"{image_name}_metrics_"
            f"threshold_{threshold_string}.txt"
        )

        save_metrics(
            metrics,
            threshold,
            image_name,
            model_path,
            metrics_path
        )

        print(
            f"Features detected: "
            f"{metrics['num_features']}"
        )

        print(
            f"Valid features: "
            f"{metrics['valid_features']}"
        )

        print(
            f"Average area: "
            f"{metrics['average_area']:.2f} pixels"
        )

        print(
            f"Total area: "
            f"{metrics['total_area']:.2f} pixels"
        )

        print(
            f"Area fraction: "
            f"{metrics['area_fraction']:.2f}%"
        )

    summary_path = os.path.join(
        output_dir,
        f"{image_name}_threshold_summary.csv"
    )

    save_threshold_summary(
        all_results,
        summary_path
    )

    print(
        "\nAnalysis complete."
    )

    print(
        f"All results saved to:\n"
        f"{output_dir}"
    )


main()

CrispTEM Feature Detector
Selected model:
D:/Rajat/CrispTEM/RAW+ masked images/best_model.pth
Selected TEM image:
D:/Rajat/CrispTEM/RAW+ masked images/Image_1.tif

Loading trained model...
Final trained model loaded.
Training epoch: 23
Validation loss: 0.02065719667977343
Using device: cuda
GPU: NVIDIA GeForce GTX 1080

Generating probability map...
Probability map saved to:
D:/Rajat/CrispTEM/RAW+ masked images/feature_model_20260916_190659\feature_analysis_Image_1_20260918_160721\Image_1_probability_map.png

Processing threshold: 0.10
Features detected: 317
Valid features: 314
Average area: 2060.29 pixels
Total area: 646932.50 pixels
Area fraction: 3.86%

Analysis complete.
All results saved to:
D:/Rajat/CrispTEM/RAW+ masked images/feature_model_20260916_190659\feature_analysis_Image_1_20260918_160721
